## Import Libraries

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import gc
import glob
from cv2 import imread, imwrite
import pandas as pd

## Definitions

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 1
BATCH_SIZE = 8
EPOCHS = 150
LEARNING_RATE = 1e-4
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2, 
    patience=3, 
    min_lr=1e-6
)


IMAGE_DIR = "Images path"
MASK_DIR = "Masks path"

## Prepare Image and Mask Sets

In [ ]:
image_paths = sorted([
    os.path.join(IMAGE_DIR, fname)
    for fname in os.listdir(IMAGE_DIR)
])

mask_paths = sorted([
    os.path.join(MASK_DIR, fname)
    for fname in os.listdir(MASK_DIR)
])

assert len(image_paths) == len(mask_paths), "Images and masks count mismatch"

train_images, val_images, train_masks, val_masks = train_test_split(
    image_paths,
    mask_paths,
    test_size=0.2,
    random_state=2026
)


## Create Tensorflow Datasets

In [ ]:
def load_image(image_path, mask_path):
    # Read image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.rgb_to_grayscale(image)
    image = tf.cast(image, tf.float32) / 255.0
    # Read mask
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.cast(mask > 127, tf.float32)
    return image, mask

def create_dataset(image_list, mask_list, batch_size):
    dataset = tf.data.Dataset.from_tensor_slices((image_list, mask_list))
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(dataset.cardinality())
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
train_dataset = create_dataset(train_images, train_masks, BATCH_SIZE)
val_dataset = create_dataset(val_images, val_masks, BATCH_SIZE)

## Define U-Net Model

In [ ]:
def conv_block(inputs, filters):
    x = layers.Conv2D(filters, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    return x

def encoder_block(inputs, filters):
    x = conv_block(inputs, filters)
    x = layers.Dropout(0.2)(x)
    p = layers.MaxPooling2D((2, 2))(x)
    return x, p

def decoder_block(inputs, skip_features, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same")(inputs)
    x = layers.Concatenate()([x, skip_features])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(224, 224, 1), n_ker=32):
    inputs = layers.Input(input_shape)

    # Encoder
    s1, p1 = encoder_block(inputs, n_ker)
    s2, p2 = encoder_block(p1, 2*n_ker)
    s3, p3 = encoder_block(p2, 4*n_ker)
    s4, p4 = encoder_block(p3, 8*n_ker)
    s5, p5 = encoder_block(p4, 16*n_ker)
        
    # Bottleneck
    b1 = conv_block(p5, 32*n_ker)
    b1 = layers.Dropout(0.5)(b1)
    # Decoder
    d1 = decoder_block(b1, s5, 16*n_ker)
    d2 = decoder_block(d1, s4, 8*n_ker)
    d3 = decoder_block(d2, s3, 4*n_ker)
    d4 = decoder_block(d3, s2, 2*n_ker)
    d5 = decoder_block(d4, s1, 1*n_ker)
    
    outputs = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(d5)

    model = Model(inputs, outputs, name="U-Net")

    return model

model = build_unet()
model.summary()

## Define loss functions

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)

    intersection = tf.reduce_sum(y_true_f * y_pred_f)

    return (
        2.0 * intersection + smooth
    ) / (
        tf.reduce_sum(y_true_f) +
        tf.reduce_sum(y_pred_f) +
        smooth
    )

def mean_iou(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    total = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    union = total - intersection
    
    iou = (intersection + smooth) / (union + smooth)
    return tf.reduce_mean(iou)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)


## Specify callbacks and compile model

In [ ]:
callbacks = [
    ModelCheckpoint(
        "save_model_path.keras",
        save_best_only=True,
        monitor="val_dice_coefficient",
        mode='max'
    ),
    EarlyStopping(
        monitor="val_dice_coefficient",
        patience=50,
        restore_best_weights=True,
        mode='max'
    )
]

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss=bce_dice_loss,
    metrics=[
        dice_coefficient, 
        mean_iou
    ]
)

## Train U-Net Model

In [ ]:
tf.keras.backend.clear_session()
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
vdc = history.history['val_dice_coefficient']
print(max(vdc), vdc.index(max(vdc))+1)


In [ ]:
if 'model' in locals():
    del model
gc.collect()

In [ ]:
model.save("best_model.keras")

print("Training completed.")

## Prediction on Test and Val Sets

In [ ]:
model = tf.keras.models.load_model("best_model.keras", custom_objects={'bce_dice_loss': bce_dice_loss, 'dice_coefficient': dice_coefficient, 'mean_iou': mean_iou})

In [ ]:
val_masks[0]

In [ ]:
test_image_dir = 'Test images path'
test_mask_dir = 'Test masks path'
image_paths = sorted(glob.glob(os.path.join(test_image_dir, '*.jpg')))
mask_paths = sorted(glob.glob(os.path.join(test_mask_dir, '*.png')))

In [ ]:
#FOR TEST SET
dice_scores = []
for img_path, mask_path in zip(image_paths, mask_paths):
    img = imread(img_path,0) / 255.0
    img_input = np.expand_dims(img, axis=0) 
    true_mask = imread(mask_path,0)
    true_mask = (true_mask > 0.5).astype(np.float32)
    pred_mask = model.predict(img_input, verbose=0)[0]
    pred_mask_binary = (pred_mask > 0.5).astype(np.float32)
    score = dice_coefficient(true_mask, pred_mask_binary)
    imwrite("Path to predicted masks/"+img_path.split("\\")[1], pred_mask*255)
    dice_scores.append(score)

In [ ]:
#FOR VAL_SET
dice_scores = []
for img_path, mask_path in zip(val_images, val_masks):
    img = imread(img_path,0) / 255.0
    img_input = np.expand_dims(img, axis=0) 
    true_mask = imread(mask_path,0)
    true_mask = (true_mask > 0.5).astype(np.float32)
    pred_mask = model.predict(img_input, verbose=0)[0]
    pred_mask_binary = (pred_mask > 0.5).astype(np.float32)
    score = dice_coefficient(true_mask, pred_mask_binary)
    imwrite("D:/KidneySegmentation/New_Data/Predicted_Masks/"+img_path.split("/")[4], pred_mask*255)
    dice_scores.append(score)

In [ ]:
mean_dice = np.median(dice_scores)
print(f"Mean Dice Coefficient across test set: {mean_dice:.4f}")

In [ ]:
dlist = list(np.array(dice_scores))
df = pd.DataFrame({'Dice_Coefficient': dlist, 'Image_Name': val_images})
df.head()

In [ ]:
df.to_csv("Path to save csv", index=False)